## Project Title

SQL Analysis of UK Online Retail Transactions

### Project Overview

#### Objective
The goal of this analysis is to explore transactional data from an online retail business to understand revenue patterns, customer behavior, and product performance.

Using SQL, the analysis focuses on data cleaning, revenue analysis, customer segmentation, and identification of key business drivers. The insights derived from this analysis could support decision-making in areas such as marketing, customer retention, and sales strategy.

### Dataset Description

#### Dataset
The dataset contains transactional records from a UK-based online retailer. Each row represents a single product line item within an invoice and includes information about the invoice number, product, quantity sold, unit price, customer identifier, country, and transaction date.

### 1. Load CSV into a SQL table

This step converts the CSV file into a format that SQL understands.

Example (Python → SQL table):

In [1]:
import pandas as pd

retail_df = pd.read_csv("online_retail.csv")

In [2]:
import pandas as pd
import sqlite3

# Create a connection to a SQLite database (or connect to an existing one)
conn = sqlite3.connect("online_retail.db")

retail_df.to_sql(
    name="online_retail",
    con=conn,
    if_exists="replace",
    index=False
)

# Optional: close the connection when done
#conn.close()

541909

### 2. Data Understanding & Scope

In [4]:
SELECT *
FROM retail_df
LIMIT 10;


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,12/1/2010 8:26,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,12/1/2010 8:26,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,12/1/2010 8:28,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,12/1/2010 8:28,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,12/1/2010 8:34,1.69,13047.0,United Kingdom


In [6]:
# Rename the DataFrame 'retail_df' to 'online_retail'
online_retail = retail_df.copy()

display(online_retail.head())

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [7]:
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT InvoiceNo) AS total_invoices,
    COUNT(DISTINCT CustomerID) AS total_customers,
    COUNT(DISTINCT Country) AS total_countries
FROM online_retail;


,total_rows,total_invoices,total_customers,total_countries
0,541909,25900,4372,38


This query summarizes the dataset by showing the total number of transaction records, unique invoices, unique customers, and the geographic scope of the business across different countries.

### 3. Data Quality & Cleaning

#### Data Cleaning Strategy
To ensure consistency across all analyses, I create a clean analytical view that excludes cancelled invoices, non-positive quantities, and non-positive unit prices. This approach improves reproducibility and mirrors best practices in analytical SQL workflows.
#### 3.1 Remove cancellations

Cancellations have `InvoiceNo` starting with 'C'

In [8]:
SELECT *
FROM online_retail
WHERE InvoiceNo LIKE 'C%';


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,C536379,D,Discount,-1,12/1/2010 9:41,27.50,14527.0,United Kingdom
1,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,12/1/2010 9:49,4.65,15311.0,United Kingdom
2,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,12/1/2010 10:24,1.65,17548.0,United Kingdom
3,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,12/1/2010 10:24,0.29,17548.0,United Kingdom
4,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,12/1/2010 10:24,0.29,17548.0,United Kingdom
...,...,...,...,...,...,...,...,...
9283,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,12/9/2011 9:57,0.83,14397.0,United Kingdom
9284,C581499,M,Manual,-1,12/9/2011 10:28,224.69,15498.0,United Kingdom
9285,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,12/9/2011 11:57,10.95,15311.0,United Kingdom
9286,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,12/9/2011 11:58,1.25,17315.0,United Kingdom


#### 3.2 Create a clean CTE

In [10]:
WITH clean_retail AS (
    SELECT *
    FROM online_retail
    WHERE InvoiceNo NOT LIKE 'C%'
      AND Quantity > 0
      AND UnitPrice > 0
)
SELECT *
FROM clean_retail
LIMIT 10;

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,12/1/2010 8:26,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,12/1/2010 8:26,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,12/1/2010 8:28,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,12/1/2010 8:28,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,12/1/2010 8:34,1.69,13047.0,United Kingdom


#### Why do I need the above conditions during the creation of a view?

**Quantity > 0** — removing returns and data errors

1. A negative Quantity usually represents:
- Returned items
- Cancelled orders

2. A Quantity = 0 means:
- No actual items were sold
- No contribution to revenue or demand

✅ Keeping only Quantity > 0 ensures you analyze items that were actually sold.


**UnitPrice > 0** — removing free or invalid items

1. A zero or negative UnitPrice typically indicates:
- Free samples
- Promotional giveaways
- Data entry errors
- System placeholders

✅ Keeping only UnitPrice > 0 ensures every transaction has a real monetary value.

A cleaned dataset is now available for analysis, providing a reliable foundation for revenue, product, and customer-level insights.

### 4. Create a revenue column
#### 4.1 Test revenue calculation

##### Key Assumptions

- Revenue is calculated as `Quantity × UnitPrice`.
- Invoices with an `InvoiceNo` starting with 'C' represent cancellations and are excluded from revenue calculations.
- Transactions with non-positive quantity or unit price are considered invalid for sales analysis.
- Customer-level analysis excludes transactions with missing customer identifiers.


In [11]:
WITH clean_retail AS (
    SELECT *
    FROM online_retail
    WHERE InvoiceNo NOT LIKE 'C%'
      AND Quantity > 0
      AND UnitPrice > 0
)
SELECT
    Quantity,
    UnitPrice,
    Quantity * UnitPrice AS revenue
FROM clean_retail
LIMIT 10;

,Quantity,UnitPrice,revenue
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34
5,2,7.65,15.30
6,6,4.25,25.50
7,6,1.85,11.10
8,6,1.85,11.10
9,32,1.69,54.08


### 5. Core Business Analysis
#### 5.1 Overall Revenue Analysis
The first business metric analyzed is total revenue, which provides a high-level view of the company’s sales performance over the observed time period.

In [12]:
WITH clean_retail AS (
    SELECT *
    FROM online_retail
    WHERE InvoiceNo NOT LIKE 'C%'
      AND Quantity > 0
      AND UnitPrice > 0
)
SELECT
    SUM(Quantity * UnitPrice) AS total_revenue
FROM clean_retail;

,total_revenue
0,1.066668e+07


The total revenue indicates substantial sales volume over the year, suggesting a healthy level of customer demand. This metric serves as a baseline for more granular analysis.

#### 5.2 Monthly revenue trend
To identify temporal patterns and seasonality, I analyze revenue aggregated at a monthly level.

In [14]:
WITH clean_retail AS (
    SELECT
        *,
        Quantity * UnitPrice AS revenue,
        STRPTIME(InvoiceDate, '%m/%d/%Y %H:%M') AS invoice_ts
    FROM online_retail
    WHERE InvoiceNo NOT LIKE 'C%'
      AND Quantity > 0
      AND UnitPrice > 0
)
SELECT
    DATE_TRUNC('month', invoice_ts) AS month,
    ROUND(SUM(revenue), 2) AS monthly_revenue
FROM clean_retail
GROUP BY month
ORDER BY month;

,month,monthly_revenue
0,2010-12-01 00:00:00+00:00,823746.14
1,2011-01-01 00:00:00+00:00,691364.56
2,2011-02-01 00:00:00+00:00,523631.89
3,2011-03-01 00:00:00+00:00,717639.36
4,2011-04-01 00:00:00+00:00,537808.62
5,2011-05-01 00:00:00+00:00,770536.02
6,2011-06-01 00:00:00+00:00,761739.90
7,2011-07-01 00:00:00+00:00,719221.19
8,2011-08-01 00:00:00+00:00,759138.38
9,2011-09-01 00:00:00+00:00,1058590.17


Revenue shows clear seasonal patterns, with noticeable increases toward the end of the year. This suggests that holiday periods play an important role in driving sales and should be considered in marketing and inventory planning.

#### 5.3 Country-level performance

In [15]:
WITH clean_retail AS (
    SELECT
        *,
        Quantity * UnitPrice AS revenue
    FROM online_retail
    WHERE InvoiceNo NOT LIKE 'C%'
      AND Quantity > 0
      AND UnitPrice > 0
),
country_revenue AS (
    SELECT
        Country,
        SUM(revenue) AS country_revenue
    FROM clean_retail
    GROUP BY Country
)
SELECT
    Country,
    ROUND(country_revenue, 2) AS country_revenue,
    ROUND(
        country_revenue * 100.0 / SUM(country_revenue) OVER (),
        2
    ) AS contribution_pct
FROM country_revenue
ORDER BY country_revenue DESC;


,Country,country_revenue,contribution_pct
0,United Kingdom,9025222.08,84.61
1,Netherlands,285446.34,2.68
2,EIRE,283453.96,2.66
3,Germany,228867.14,2.15
4,France,209715.11,1.97
5,Australia,138521.31,1.30
6,Spain,61577.11,0.58
7,Switzerland,57089.90,0.54
8,Belgium,41196.34,0.39
9,Sweden,38378.33,0.36


The contribution analysis shows that revenue is heavily concentrated in the United Kingdom, which accounts for the vast majority of total sales. The UK alone contributes approximately 84.61% of the overall revenue, while all other countries individually represent only a small fraction. This indicates a strong dependence on the domestic market and limited diversification across international markets.

### 6. Product analytics (business-relevant)
#### 6.1 Top products by revenue
##### Customer Lifetime Value Analysis
To better understand how revenue is distributed across customers, I calculate customer lifetime value (CLV) as the total revenue generated by each customer over the observed time period. CLV helps identify high-value customers and assess the degree of revenue concentration, which is critical for customer retention and targeted marketing strategies.

In [19]:
WITH clean_retail AS (
    SELECT
        *,
        Quantity * UnitPrice AS revenue
    FROM online_retail
    WHERE InvoiceNo NOT LIKE 'C%'
      AND Quantity > 0
      AND UnitPrice > 0
) 

SELECT 
    CustomerID,
    ROUND(SUM(revenue), 2) AS lifetime_value
FROM clean_retail
WHERE CustomerID IS NOT NULL
GROUP BY CustomerID
ORDER BY lifetime_value DESC;   

,CustomerID,lifetime_value
0,14646.0,280206.02
1,18102.0,259657.30
2,17450.0,194550.79
3,16446.0,168472.50
4,14911.0,143825.06
...,...,...
4333,16878.0,13.30
4334,17956.0,12.75
4335,16454.0,6.90
4336,14792.0,6.20


Customer lifetime value is heavily concentrated within a small group of customers, indicating that retaining these high-value customers is essential for maintaining revenue. Losing even a few of them could significantly affect overall sales performance. Implementing targeted retention strategies—such as personalized offers or loyalty programs—focused on high-value customers can meaningfully enhance long-term revenue stability.

#### 6.2 Purchase Frequency
To complement customer lifetime value, I analyze purchase frequency by measuring how often each customer places an order. Purchase frequency helps distinguish between customers who generate high revenue through repeated purchases and those who contribute mainly through high-value single transactions. This provides additional insight into customer engagement and _loyalty._

In [17]:
WITH clean_retail AS (
    SELECT
        *,
        Quantity * UnitPrice AS revenue
    FROM online_retail
    WHERE InvoiceNo NOT LIKE 'C%'
      AND Quantity > 0
      AND UnitPrice > 0
)  

SELECT
    CustomerID,
    COUNT(DISTINCT InvoiceNo) AS num_orders,
    ROUND(SUM(revenue), 2) AS total_spend
FROM clean_retail
WHERE CustomerID IS NOT NULL
GROUP BY CustomerID
ORDER BY num_orders DESC;

,CustomerID,num_orders,total_spend
0,12748.0,209,33719.73
1,14911.0,201,143825.06
2,17841.0,124,40991.57
3,13089.0,97,58825.83
4,14606.0,93,12156.65
...,...,...,...
4333,15678.0,1,352.70
4334,15327.0,1,208.75
4335,15853.0,1,110.80
4336,18019.0,1,38.45


Customers who purchase more frequently contribute disproportionately to total revenue, highlighting the importance of customer engagement and repeat buying in driving business performance. Additionally, promoting repeat purchases through loyalty incentives or targeted campaigns can boost customer lifetime value and help create more stable, predictable revenue over time.

#### 6.3 Repeat vs One-Time Customers

##### Customer Retention Analysis
To further understand customer behavior, I classify customers into one-time and repeat buyers based on the number of distinct invoices associated with each customer. This analysis helps assess customer retention and identifies whether revenue is primarily driven by long-term relationships or single-purchase customers.

In [19]:
WITH clean_retail AS (
    SELECT
        *,
        Quantity * UnitPrice AS revenue
    FROM online_retail
    WHERE InvoiceNo NOT LIKE 'C%'
      AND Quantity > 0
      AND UnitPrice > 0
)  

SELECT
    CASE
        WHEN order_count = 1 THEN 'One-time'
        ELSE 'Repeat'
    END AS customer_type,
    COUNT(*) AS customers
FROM (
    SELECT
        CustomerID,
        COUNT(DISTINCT InvoiceNo) AS order_count
    FROM clean_retail
    WHERE CustomerID IS NOT NULL
    GROUP BY CustomerID
) subq
GROUP BY customer_type;


,customer_type,customers
0,Repeat,2845
1,One-time,1493


The dominance of repeat customers suggests that the company has been successful in retaining customers over time. This reduces reliance on constant customer acquisition and indicates a more stable and predictable revenue model. Continuing to maintain and enhance retention strategies can further increase customer lifetime value and support sustained long-term revenue growth.

#### 6.4 Pareto Analysis (Revenue Concentration)

##### Revenue Concentration Analysis (Pareto Principle)
To assess how evenly revenue is distributed across customers, I perform a Pareto analysis by ranking customers based on their total revenue contribution and calculating the cumulative share of revenue. This analysis helps determine whether a small proportion of customer accounts for a large share of total sales, which is important for understanding revenue concentration and dependency risk.

In [21]:
WITH clean_retail AS (
    SELECT
        *,
        Quantity * UnitPrice AS revenue
    FROM online_retail
    WHERE InvoiceNo NOT LIKE 'C%'
      AND Quantity > 0
      AND UnitPrice > 0
),
customer_revenue AS (
    SELECT
        CustomerID,
        SUM(revenue) AS total_revenue
    FROM clean_retail
    WHERE CustomerID IS NOT NULL
    GROUP BY CustomerID
),
ranked_customers AS (
    SELECT
        CustomerID,
        total_revenue,
        SUM(total_revenue) OVER (
            ORDER BY total_revenue DESC
        ) AS cumulative_revenue,
        SUM(total_revenue) OVER () AS overall_revenue
    FROM customer_revenue
)
SELECT
    CustomerID,
    ROUND(total_revenue, 2) AS customer_revenue,
    ROUND(cumulative_revenue / overall_revenue * 100, 2) AS cumulative_revenue_pct
FROM ranked_customers
ORDER BY customer_revenue DESC;

,CustomerID,customer_revenue,cumulative_revenue_pct
0,14646.0,280206.02,3.14
1,18102.0,259657.30,6.06
2,17450.0,194550.79,8.24
3,16446.0,168472.50,10.13
4,14911.0,143825.06,11.75
...,...,...,...
4333,16878.0,13.30,100.00
4334,17956.0,12.75,100.00
4335,16454.0,6.90,100.00
4336,14792.0,6.20,100.00


Revenue is heavily concentrated among a small group of top customers, making their retention essential for maintaining overall sales. Although many customers are active, a limited number contribute a disproportionately large share of revenue. Focusing retention efforts on these high-value customers can help mitigate revenue risk and strengthen long-term business stability.

#### Conclusion
This analysis examined transactional data from an online retail business using SQL to understand revenue patterns, product performance, and customer behavior. The results show that revenue is driven by a combination of strong customer retention and a relatively small group of high-value customers. Seasonal trends and geographic concentration further highlight opportunities for targeted marketing and strategic growth. Overall, the analysis demonstrates how structured SQL analysis can support data-driven business decisions.

#### Limitations and Future Work
The analysis is limited to historical transactional data and does not account for marketing activities, customer demographics, or profit margins. Future work could extend this analysis by incorporating predictive modeling, customer segmentation, or churn analysis using Python or R.